<a href="https://colab.research.google.com/github/IsaacLeng/Crypto-Chain-Data-Fetcher/blob/main/ETH_SOL_TPS_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install web3 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.3/340.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 74.3 MB/s eta 0:00:00


In [5]:
from web3 import Web3
import datetime

# eth_url = "https://rpc.ankr.com/eth"  # Backup Node 1 / 备用节点 1
eth_url = "https://ethereum-rpc.publicnode.com" # Backup Node 2 / 备用节点 2

w3 = Web3(Web3.HTTPProvider(eth_url))

# Check connection status / 检查连接状态
if w3.is_connected():
    # Get the latest block / 获取最新区块
    latest_block = w3.eth.get_block('latest')

    # Extracting key data / 提取关键数据
    block_num = latest_block.number
    block_time = datetime.datetime.fromtimestamp(latest_block.timestamp)
    tx_count = len(latest_block.transactions)
    block_size = latest_block.size

    print("=== Ethereum latest block 以太坊最新区块 ===")
    print(f"Block height / 区块高度: {block_num}")
    print(f"Block time / 出块时间: {block_time}")
    print(f"Includes number of transactions / 包含交易数: {tx_count} 笔")
    print(f"Block size / 区块大小: {block_size / 1024:.2f} KB")
else:
    print("Ethereum node connection failed, please try a different one. / 以太坊节点连接失败，请尝试更换 eth_url")

=== Ethereum latest block 以太坊最新区块 ===
Block height / 区块高度: 24666133
Block time / 出块时间: 2026-03-15 23:18:59
Includes number of transactions / 包含交易数: 259 笔
Block size / 区块大小: 278.76 KB


In [12]:
import requests
import time

# Solana Mainnet Beta Public Nodes / Solana 主网 Beta 公共节点
sol_url = "https://api.mainnet-beta.solana.com"
headers = {"Content-Type": "application/json"}

print("\n=== Solana latest block / Solana 最新区块 ===")

# Step 1: Obtain the latest Slot (block height concept in Solana) / 第一步：获取当前最新的 Slot (Solana 中的区块高度概念)
payload_slot = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "getSlot"
}
response_slot = requests.post(sol_url, json=payload_slot, headers=headers).json()
latest_slot = response_slot.get('result')

print(f"Latest Slot Height / 最新 Slot 高度: {latest_slot}")

# Step 2: Obtain block details based on the Slot / 第二步：根据 Slot 获取区块详情
payload_block = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "getBlock",
    "params": [
        latest_slot,
        {
            "encoding": "json",
            "maxSupportedTransactionVersion": 0,
            "transactionDetails": "signatures" # To expedite the response, we only retrieve the list of transaction signatures. / 为了加快响应，我们只获取交易签名列表
        }
    ]
}

# Solana produces blocks too quickly; sometimes the latest slot hasn't been fully confirmed yet. We can wait a second or fetch the previous slot. / Solana 出块太快，有时最新的 slot 还没完全确认，我们可以稍微等一秒或抓取前一个 slot
try:
    response_block = requests.post(sol_url, json=payload_block, headers=headers).json()
    if 'result' in response_block and response_block['result'] is not None:
        block_data = response_block['result']
        signatures_list = block_data.get('signatures', [])
        tx_count = len(signatures_list)

        print(f"Includes number of transactions / 包含交易数: {tx_count} 笔")
    else:
        print("This block is still being confirmed. Please try again or fetch an earlier slot. / 该区块仍在确认中，请重试或抓取稍早的 Slot。")
except Exception as e:
    print(f"Fetch failed / 抓取失败: {e}")


=== Solana latest block / Solana 最新区块 ===
Latest Slot Height / 最新 Slot 高度: 406674942
Includes number of transactions / 包含交易数: 1430 笔
